[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/apis-avanzadas/01-extended-thinking.ipynb)

# Extended Thinking: Razonamiento Profundo con Claude

> Aprende a usar el razonamiento extendido de Claude para resolver problemas
> complejos con mayor precisión: matemáticas, lógica y planificación estratégica.

## Objetivos
- Entender la diferencia entre respuesta directa y razonamiento extendido
- Configurar `budget_tokens` según la complejidad de la tarea
- Comparar precisión con y sin thinking en problemas matemáticos
- Implementar streaming con bloques de thinking

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic --quiet

## 2. Configuración y primer ejemplo

In [ ]:
import anthropic
import time

client = anthropic.Anthropic()

def responder_con_thinking(pregunta: str, budget_tokens: int = 8000) -> dict:
    """Llama a Claude con Extended Thinking y devuelve thinking + respuesta."""
    inicio = time.perf_counter()
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=budget_tokens + 2000,
        thinking={"type": "enabled", "budget_tokens": budget_tokens},
        messages=[{"role": "user", "content": pregunta}]
    )
    latencia = time.perf_counter() - inicio

    thinking_texto = ""
    respuesta_texto = ""
    for bloque in resp.content:
        if bloque.type == "thinking":
            thinking_texto = bloque.thinking
        elif bloque.type == "text":
            respuesta_texto = bloque.text

    return {
        "thinking": thinking_texto,
        "respuesta": respuesta_texto,
        "tokens_thinking": len(thinking_texto.split()),
        "latencia_s": round(latencia, 2),
        "usage": resp.usage
    }

def responder_sin_thinking(pregunta: str) -> dict:
    """Llama a Claude sin thinking para comparar."""
    inicio = time.perf_counter()
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,
        messages=[{"role": "user", "content": pregunta}]
    )
    return {
        "respuesta": resp.content[0].text,
        "latencia_s": round(time.perf_counter() - inicio, 2)
    }

print("Cliente configurado. Modelo: claude-sonnet-4-6")
print("Extended Thinking requiere claude-sonnet-4-6 o superior.")

## 3. Problema matemático: con y sin thinking

In [ ]:
PROBLEMA_MATEMATICO = """
Una empresa produce 3 productos (A, B, C) con los siguientes datos:
- Producto A: beneficio 5€/ud, requiere 2h máquina, 1h mano de obra
- Producto B: beneficio 4€/ud, requiere 1h máquina, 2h mano de obra  
- Producto C: beneficio 3€/ud, requiere 1h máquina, 1h mano de obra

Restricciones semanales:
- Máquina disponible: 40 horas
- Mano de obra disponible: 50 horas
- Demanda máxima A: 15 unidades, B: 20 unidades, C: 30 unidades

¿Cuántas unidades de cada producto maximizan el beneficio?
Muestra el beneficio total óptimo.
"""

print("Probando SIN extended thinking...")
sin_thinking = responder_sin_thinking(PROBLEMA_MATEMATICO)
print(f"Latencia: {sin_thinking['latencia_s']}s")
print(f"Respuesta:\n{sin_thinking['respuesta']}")

print("\n" + "="*60)
print("Probando CON extended thinking (budget: 8000 tokens)...")
con_thinking = responder_con_thinking(PROBLEMA_MATEMATICO, budget_tokens=8000)
print(f"Latencia: {con_thinking['latencia_s']}s")
print(f"Palabras en el razonamiento: ~{con_thinking['tokens_thinking']}")
print(f"\nRazonamiento interno (primeros 500 chars):")
print(con_thinking['thinking'][:500] + "...")
print(f"\nRespuesta final:\n{con_thinking['respuesta']}")

## 4. Análisis de código con razonamiento

In [ ]:
CODIGO_CON_BUGS = """
def calcular_estadisticas(datos):
    n = len(datos)
    media = sum(datos) / n
    
    varianza = sum((x - media) ** 2 for x in datos) / n  # ¿sesgo?
    desviacion = varianza ** 0.5
    
    datos_ordenados = sorted(datos)
    if n % 2 == 0:
        mediana = datos_ordenados[n // 2]  # ¿correcto para n par?
    else:
        mediana = datos_ordenados[n // 2]
    
    moda = max(datos, key=datos.count)
    
    return {"media": media, "varianza": varianza, 
            "desviacion": desviacion, "mediana": mediana, "moda": moda}
"""

pregunta_codigo = f"""Analiza este código Python en profundidad:
1. Identifica todos los bugs (incluyendo bugs sutiles)
2. Explica por qué cada uno es un error
3. Proporciona la versión corregida

```python
{CODIGO_CON_BUGS}
```"""

resultado = responder_con_thinking(pregunta_codigo, budget_tokens=5000)
print("=== ANÁLISIS DE CÓDIGO CON EXTENDED THINKING ===")
print(f"Latencia: {resultado['latencia_s']}s")
print("\nRazonamiento (extracto):")
print(resultado['thinking'][:600] + "...")
print("\n" + "="*50)
print("RESPUESTA FINAL:")
print(resultado['respuesta'])

## 5. Decisión estratégica con razonamiento explícito

In [ ]:
DILEMA_ESTRATEGICO = """
Soy CTO de una startup SaaS B2B con 3 años de vida. Situación actual:
- MRR: 85.000€ (crecimiento +8% mensual últimos 6 meses)
- Equipo: 12 personas (6 ingenieros, 2 ventas, 2 CS, 2 ops)
- Caja: 1.2M€ (runway ~14 meses)
- Deuda técnica: significativa, microservicios mal diseñados
- Principal queja de clientes: lentitud y fallos en producción

Opciones:
A) Refactoring total (6 meses, 2 ingenieros dedicados): frenar features
B) Contratar 2 ingenieros senior ahora (240K€/año): acelerar todo
C) Migrar a arquitectura nueva en paralelo (9 meses, todo el equipo parcialmente)
D) Lanzar en mercado EEUU ya (necesita 150K€ en marketing): crecer rápido

¿Qué combinación de acciones recomiendas y en qué orden? ¿Por qué?
"""

resultado = responder_con_thinking(DILEMA_ESTRATEGICO, budget_tokens=10000)

print("=== DECISIÓN ESTRATÉGICA CON EXTENDED THINKING ===")
print(f"Tiempo de razonamiento: {resultado['latencia_s']}s")
print(f"Profundidad del análisis: ~{resultado['tokens_thinking']} palabras")
print("\n--- RAZONAMIENTO INTERNO ---")
# Mostrar los primeros 800 chars del thinking
thinking_preview = resultado['thinking'][:800]
print(thinking_preview + "\n...")
print("\n--- RECOMENDACIÓN FINAL ---")
print(resultado['respuesta'])

print("\n=== COMPARATIVA DE USO ===")
print(f"Tokens entrada: {resultado['usage'].input_tokens}")
print(f"Tokens salida: {resultado['usage'].output_tokens}")
coste_est = (resultado['usage'].input_tokens * 3 + resultado['usage'].output_tokens * 15) / 1_000_000
print(f"Coste estimado (Sonnet): ${coste_est:.4f}")
print("\nConsejos de uso:")
print("  • budget_tokens bajo (1K-3K): problemas simples de razonamiento")
print("  • budget_tokens medio (5K-10K): análisis de código, estrategia")
print("  • budget_tokens alto (15K-32K): demostraciones matemáticas, planificación compleja")